# 🎨 Florence-2 — OCR, Description & Détection d'objets

Ce notebook utilise **Florence-2** (Microsoft) pour :
- 📝 **OCR** — Extraire le texte d'une image
- 🖼️ **Captioning** — Décrire le contenu d'une image
- 🔍 **Détection d'objets** — Localiser et identifier les objets

---

## 1. Installation des dépendances
Exécutez cette cellule une seule fois.

In [ ]:
# Installer une version compatible de transformers pour Florence-2
!pip install -q transformers==4.46.3 torch torchvision pillow einops timm
# flash_attn est optionnel (accélère sur GPU compatible) :
# !pip install flash_attn

## 2. Chargement du modèle Florence-2

In [ ]:
import torch
from transformers import AutoProcessor, AutoModelForCausalLM
from PIL import Image, ImageDraw, ImageFont
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import requests
from io import BytesIO

# Choisir le device
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"🖥️ Device utilisé : {device}")

# Charger le modèle (première exécution = téléchargement ~1.5 GB)
MODEL_ID = "microsoft/Florence-2-large"
# Alternative plus légère : "microsoft/Florence-2-base"

print(f"⏳ Chargement de {MODEL_ID}...")
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    trust_remote_code=True,
    torch_dtype=torch.float16
).to(device).eval()

processor = AutoProcessor.from_pretrained(MODEL_ID, trust_remote_code=True)
print("✅ Modèle chargé avec succès !")

## 3. Fonction utilitaire
Une seule fonction pour exécuter n'importe quelle tâche Florence-2.

In [ ]:
def run_florence(image: Image.Image, task: str, text_input: str = "") -> dict:
    """
    Exécute une tâche Florence-2 sur une image.
    
    Tâches disponibles :
      - '<OCR>'                  : Extraction de texte brut
      - '<OCR_WITH_REGION>'      : Texte + positions (bounding boxes)
      - '<CAPTION>'              : Description courte
      - '<DETAILED_CAPTION>'     : Description détaillée
      - '<MORE_DETAILED_CAPTION>': Description très détaillée
      - '<OD>'                   : Détection d'objets
      - '<DENSE_REGION_CAPTION>' : Légendes par région
      - '<REGION_PROPOSAL>'      : Propositions de régions
      - '<CAPTION_TO_PHRASE_GROUNDING>' : Localiser des éléments décrits
    """
    prompt = task + text_input
    inputs = processor(text=prompt, images=image, return_tensors="pt").to(device)
    
    # Caster les pixel_values au même dtype que le modèle (float16 sur GPU)
    model_dtype = next(model.parameters()).dtype
    if "pixel_values" in inputs:
        inputs["pixel_values"] = inputs["pixel_values"].to(model_dtype)
    
    with torch.no_grad():
        generated_ids = model.generate(
            **inputs,
            max_new_tokens=1024,
            num_beams=3,
            do_sample=False
        )
    
    generated_text = processor.batch_decode(generated_ids, skip_special_tokens=False)[0]
    result = processor.post_process_generation(
        generated_text,
        task=task,
        image_size=(image.width, image.height)
    )
    return result

## 4. Charger une image
Choisissez votre méthode :

In [ ]:
# === MÉTHODE 1 : Depuis une URL ===
url = "https://upload.wikimedia.org/wikipedia/commons/thumb/4/47/PNG_transparency_demonstration_1.png/300px-PNG_transparency_demonstration_1.png"
response = requests.get(url)
image = Image.open(BytesIO(response.content)).convert("RGB")

# === MÉTHODE 2 : Depuis un fichier local ===
# image = Image.open("chemin/vers/votre/image.jpg").convert("RGB")

# === MÉTHODE 3 : Upload dans Jupyter/Colab ===
# from google.colab import files
# uploaded = files.upload()
# filename = list(uploaded.keys())[0]
# image = Image.open(filename).convert("RGB")

# Afficher l'image
plt.figure(figsize=(8, 6))
plt.imshow(image)
plt.axis("off")
plt.title("Image chargée")
plt.show()
print(f"📐 Dimensions : {image.width} x {image.height}")

---
## 📝 5. OCR — Extraction de texte

In [ ]:
# OCR simple
result_ocr = run_florence(image, "<OCR>")
print("=" * 50)
print("📝 TEXTE EXTRAIT :")
print("=" * 50)
print(result_ocr.get("<OCR>", "Aucun texte trouvé"))

In [ ]:
# OCR avec régions (texte + positions)
result_ocr_regions = run_florence(image, "<OCR_WITH_REGION>")

data = result_ocr_regions.get("<OCR_WITH_REGION>", {})

if data and "quad_boxes" in data:
    fig, ax = plt.subplots(1, figsize=(12, 8))
    ax.imshow(image)
    
    colors = plt.cm.Set2.colors
    for i, (text, box) in enumerate(zip(data["labels"], data["quad_boxes"])):
        color = colors[i % len(colors)]
        # quad_boxes = [x1,y1, x2,y2, x3,y3, x4,y4]
        xs = [box[0], box[2], box[4], box[6], box[0]]
        ys = [box[1], box[3], box[5], box[7], box[1]]
        ax.plot(xs, ys, linewidth=2, color=color)
        ax.text(box[0], box[1] - 5, text, fontsize=8, color=color,
                bbox=dict(boxstyle="round,pad=0.2", facecolor="black", alpha=0.7))
    
    ax.axis("off")
    ax.set_title("OCR avec régions")
    plt.tight_layout()
    plt.show()
else:
    print("Aucune région de texte détectée.")

---
## 🖼️ 6. Captioning — Description d'image

In [ ]:
# Description courte
result_caption = run_florence(image, "<CAPTION>")
print("📌 Description courte :")
print(result_caption.get("<CAPTION>", ""))

print()

# Description détaillée
result_detailed = run_florence(image, "<DETAILED_CAPTION>")
print("📝 Description détaillée :")
print(result_detailed.get("<DETAILED_CAPTION>", ""))

print()

# Description très détaillée
result_more = run_florence(image, "<MORE_DETAILED_CAPTION>")
print("📖 Description très détaillée :")
print(result_more.get("<MORE_DETAILED_CAPTION>", ""))

---
## 🔍 7. Détection d'objets

In [ ]:
def plot_detections(image, result, title="Détection d'objets"):
    """Affiche les bounding boxes sur l'image."""
    fig, ax = plt.subplots(1, figsize=(12, 8))
    ax.imshow(image)
    
    colors = plt.cm.tab20.colors
    
    bboxes = result.get("bboxes", [])
    labels = result.get("labels", [])
    
    for i, (bbox, label) in enumerate(zip(bboxes, labels)):
        color = colors[i % len(colors)]
        x1, y1, x2, y2 = bbox
        w, h = x2 - x1, y2 - y1
        
        rect = patches.Rectangle(
            (x1, y1), w, h,
            linewidth=2, edgecolor=color, facecolor="none"
        )
        ax.add_patch(rect)
        ax.text(
            x1, y1 - 5, label,
            fontsize=10, fontweight="bold", color="white",
            bbox=dict(boxstyle="round,pad=0.3", facecolor=color, alpha=0.8)
        )
    
    ax.axis("off")
    ax.set_title(f"{title} — {len(bboxes)} objet(s) détecté(s)")
    plt.tight_layout()
    plt.show()
    
    # Liste des objets
    print(f"\n🔎 {len(bboxes)} objet(s) détecté(s) :")
    for i, label in enumerate(labels):
        print(f"  {i+1}. {label}")

In [ ]:
# Détection d'objets
result_od = run_florence(image, "<OD>")
plot_detections(image, result_od.get("<OD>", {}), "Détection d'objets")

In [ ]:
# Légendes denses par région
result_dense = run_florence(image, "<DENSE_REGION_CAPTION>")
plot_detections(image, result_dense.get("<DENSE_REGION_CAPTION>", {}), "Légendes par région")

---
## 🎯 8. Bonus : Grounding — Localiser un élément par description
Donnez une description en anglais et Florence-2 localise l'élément dans l'image.

In [ ]:
# Changez cette description pour chercher d'autres éléments
query = "a red car"

result_grounding = run_florence(image, "<CAPTION_TO_PHRASE_GROUNDING>", query)
data = result_grounding.get("<CAPTION_TO_PHRASE_GROUNDING>", {})

if data and "bboxes" in data and len(data["bboxes"]) > 0:
    plot_detections(image, data, f'Recherche : "{query}"')
else:
    print(f"❌ Rien trouvé pour : \"{query}\"")

---
## 🚀 9. Traitement par lot (plusieurs images)
Pour traiter un dossier entier d'images :

In [ ]:
import os
import json

def process_folder(folder_path: str, tasks: list = None) -> list:
    """
    Traite toutes les images d'un dossier.
    
    Args:
        folder_path: Chemin vers le dossier d'images
        tasks: Liste de tâches (défaut: OCR + Caption + OD)
    
    Returns:
        Liste de résultats par image
    """
    if tasks is None:
        tasks = ["<OCR>", "<CAPTION>", "<OD>"]
    
    extensions = {".jpg", ".jpeg", ".png", ".bmp", ".webp", ".tiff"}
    results = []
    
    files = sorted([
        f for f in os.listdir(folder_path)
        if os.path.splitext(f)[1].lower() in extensions
    ])
    
    print(f"📂 {len(files)} image(s) trouvée(s) dans {folder_path}")
    
    for i, filename in enumerate(files, 1):
        filepath = os.path.join(folder_path, filename)
        print(f"\n[{i}/{len(files)}] Traitement de {filename}...")
        
        img = Image.open(filepath).convert("RGB")
        img_results = {"filename": filename}
        
        for task in tasks:
            result = run_florence(img, task)
            img_results[task] = result.get(task, {})
            
            # Afficher un aperçu
            if task == "<OCR>":
                text = result.get(task, "")
                print(f"  📝 Texte : {text[:100]}{'...' if len(str(text)) > 100 else ''}")
            elif task == "<CAPTION>":
                print(f"  🖼️ Description : {result.get(task, '')}")
            elif task == "<OD>":
                labels = result.get(task, {}).get("labels", [])
                print(f"  🔍 Objets : {', '.join(labels) if labels else 'aucun'}")
        
        results.append(img_results)
    
    return results

# Exemple d'utilisation :
# results = process_folder("./mes_images/")
# 
# Sauvegarder les résultats en JSON :
# with open("resultats.json", "w", encoding="utf-8") as f:
#     json.dump(results, f, ensure_ascii=False, indent=2)

print("\n✅ Fonction prête ! Décommentez les lignes ci-dessus pour l'utiliser.")

---
## 📋 Récapitulatif des tâches disponibles

| Tâche | Code | Description |
|-------|------|-------------|
| OCR simple | `<OCR>` | Texte brut |
| OCR + positions | `<OCR_WITH_REGION>` | Texte + bounding boxes |
| Caption courte | `<CAPTION>` | 1 phrase |
| Caption détaillée | `<DETAILED_CAPTION>` | Plusieurs phrases |
| Caption très détaillée | `<MORE_DETAILED_CAPTION>` | Paragraphe complet |
| Détection d'objets | `<OD>` | Labels + bounding boxes |
| Légendes par région | `<DENSE_REGION_CAPTION>` | Description de chaque zone |
| Grounding | `<CAPTION_TO_PHRASE_GROUNDING>` | Localiser un élément décrit |